# Лабораторная работа №3
## Арифметические вычисления — вариант 19

**Ассемблер:** FASM (Flat Assembler)  
**Платформа:** Linux x86-64  
**Примечание:** все программы в работе написаны **только на FASM**. C/NASM не используются.

### Цель работы
1. Реализовать программу, которая получает символ из командной строки и выводит его ASCII-код.
2. Реализовать арифметическое выражение для варианта 19.
3. Передавать входные данные через командную строку.
4. Проверить правильность вычислений на нескольких наборах данных.


## 1. Постановка задачи

Согласно заданию, символ и значения `a`, `b`, `c` передаются исполняемой программе как параметры командной строки.

Для варианта 19 необходимо вычислить выражение:

$$(((((a+c)-b)\cdot c)\cdot c)-c)$$

или:

$$ (a+c-b)c^2-c $$

Все программы ниже являются **самостоятельными ELF64-программами на FASM** и используют системные вызовы Linux, поэтому для их сборки не требуется C или другая библиотека.


## 2. Проверка варианта 19

Используем алгоритм генерации из задания с `Variant = 19`.


In [ ]:
import random as rd

Variant = 19
rd.seed(Variant)

set_operations = ['+', '-', '*', '/']
set_operands = ['a', 'b', 'c']
count_operations = rd.randint(3, 5)

expression = set_operands[rd.randint(0, len(set_operands)-1)]

for i in range(count_operations):
    current_operation = set_operations[rd.randint(0, len(set_operations)-1)]
    current_operand = set_operands[rd.randint(0, len(set_operands)-1)]
    expression = '(' + expression + current_operation + current_operand + ')'

print("Вариант:", Variant)
print("Выражение:", expression)


## 3. ASCII-код символа — FASM

Программа получает первый символ из `argv[1]`, преобразует его в ASCII-код и выводит результат.

Файл `ascii.asm`:


In [ ]:
%%writefile ascii.asm
format ELF64 executable 3
entry start

segment readable executable

start:
    ; argc
    mov rax, [rsp]
    cmp rax, 2
    jl .exit

    ; argv[1]
    mov rsi, [rsp + 16]

    ; Первый символ
    movzx eax, byte [rsi]

    ; Печатаем строку
    push rax
    mov rsi, msg
    mov edx, msg_len
    call print_string
    pop rax

    ; Печатаем ASCII-код
    call print_uint

    ; Перевод строки
    mov rsi, newline
    mov edx, 1
    call print_string

.exit:
    mov eax, 60
    xor edi, edi
    syscall

; ---------------------------------------
; print_string
; RSI = адрес строки
; EDX = длина
; ---------------------------------------
print_string:
    mov eax, 1
    mov edi, 1
    syscall
    ret

; ---------------------------------------
; print_uint
; EAX = беззнаковое число
; ---------------------------------------
print_uint:
    mov r8, num_buffer
    add r8, 31

    xor ecx, ecx
    mov ebx, 10

.convert:
    xor edx, edx
    div ebx
    add dl, '0'
    dec r8
    mov [r8], dl
    inc ecx
    test eax, eax
    jnz .convert

    mov rsi, r8
    mov edx, ecx
    call print_string
    ret

segment readable writeable

msg db 'ASCII-code: '
msg_len = $ - msg
newline db 10
num_buffer rb 32


### Сборка и запуск

В терминале Linux:

```bash
fasm ascii.asm ascii
chmod +x ascii
./ascii A
```

Ожидаемый результат:

```text
ASCII-code: 65
```

Для символа `a`:

```text
ASCII-code: 97
```


## 4. Арифметическое выражение — FASM

В этой программе **нет C-кода и нет вызовов `atoi`, `printf` или других библиотек**.

Параметры командной строки самостоятельно преобразуются из строк в целые числа. После этого выполняется:

1. `a + c`
2. `(a + c) - b`
3. `((a + c) - b) * c`
4. `(((a + c) - b) * c) * c`
5. `... - c`

Файл `calc.asm`:


In [ ]:
%%writefile calc.asm
format ELF64 executable 3
entry start

segment readable executable

start:
    mov rax, [rsp]
    cmp rax, 4
    jne .input_error

    mov rsi, [rsp + 16]
    call parse_int
    mov r12d, eax

    mov rsi, [rsp + 24]
    call parse_int
    mov r13d, eax

    mov rsi, [rsp + 32]
    call parse_int
    mov r14d, eax

    ; Вариант 19:
    ; (((((a+c)-b)*c)*c)-c)
    mov eax, r12d
    add eax, r14d
    sub eax, r13d
    imul eax, r14d
    imul eax, r14d
    sub eax, r14d
    mov r15d, eax

    mov rsi, expression_msg
    mov edx, expression_msg_len
    call print_string

    mov eax, r12d
    call print_int

    mov rsi, b_msg
    mov edx, b_msg_len
    call print_string

    mov eax, r13d
    call print_int

    mov rsi, c_msg
    mov edx, c_msg_len
    call print_string

    mov eax, r14d
    call print_int

    mov rsi, result_label
    mov edx, result_label_len
    call print_string

    mov eax, r15d
    call print_int

    mov rsi, newline
    mov edx, 1
    call print_string

    jmp .exit

.input_error:
    mov rsi, usage_msg
    mov edx, usage_msg_len
    call print_string

.exit:
    mov eax, 60
    xor edi, edi
    syscall


; ---------------------------------------
; parse_int
; RSI -> строка
; EAX <- signed integer
; ---------------------------------------
parse_int:
    xor eax, eax
    xor ecx, ecx

    cmp byte [rsi], '-'
    jne .digits

    mov ecx, 1
    inc rsi

.digits:
    movzx edx, byte [rsi]
    test dl, dl
    jz .done

    cmp dl, '0'
    jb .done
    cmp dl, '9'
    ja .done

    imul eax, eax, 10
    sub edx, '0'
    add eax, edx

    inc rsi
    jmp .digits

.done:
    test ecx, ecx
    jz .return

    neg eax

.return:
    ret


; ---------------------------------------
; print_string
; RSI = адрес
; EDX = длина
; ---------------------------------------
print_string:
    mov eax, 1
    mov edi, 1
    syscall
    ret


; ---------------------------------------
; print_int
; EAX = signed 32-bit integer
; ---------------------------------------
print_int:
    test eax, eax
    jns .positive

    push rax
    mov rsi, minus
    mov edx, 1
    call print_string
    pop rax

    neg eax

.positive:
    mov r8, num_buffer
    add r8, 31

    xor ecx, ecx
    mov ebx, 10

    test eax, eax
    jnz .convert

    dec r8
    mov byte [r8], '0'
    mov ecx, 1
    jmp .write

.convert:
    xor edx, edx
    div ebx
    add dl, '0'
    dec r8
    mov [r8], dl
    inc ecx

    test eax, eax
    jnz .convert

.write:
    mov rsi, r8
    mov edx, ecx
    call print_string
    ret


segment readable writeable

expression_msg db 'Expression: (((((a+c)-b)*c)*c)-c)', 10
expression_msg_len = $ - expression_msg

b_msg db ', b = '
b_msg_len = $ - b_msg

c_msg db ', c = '
c_msg_len = $ - c_msg

result_label db ', result = '
result_label_len = $ - result_label

usage_msg db 'Usage: ./calc a b c', 10
usage_msg_len = $ - usage_msg

minus db '-'
newline db 10

num_buffer rb 32


### Сборка и запуск

В терминале Linux:

```bash
fasm calc.asm calc
chmod +x calc
./calc 2 3 4
```

Ожидаемый результат:

```text
Expression: (((((a+c)-b)*c)*c)-c)
a = 2, b = 3, c = 4, result = 44
```


## 5. Проверка вычислений

Для `a = 2`, `b = 3`, `c = 4`:

$$(((((2+4)-3)\cdot4)\cdot4)-4)$$

$$((3\cdot4)\cdot4)-4$$

$$48-4=44$$


In [ ]:
# Если FASM установлен в Linux-среде Jupyter:
!fasm calc.asm calc
!chmod +x calc
!./calc 2 3 4


### Дополнительные тесты

| a | b | c | Результат |
|---:|---:|---:|---:|
| 2 | 3 | 4 | 44 |
| 5 | 2 | 3 | 51 |
| 10 | 4 | 2 | 22 |
| 7 | 3 | 5 | 95 |

Проверка:

- `(5 + 3 - 2) * 3 * 3 - 3 = 51`
- `(10 + 2 - 4) * 2 * 2 - 2 = 30`, поэтому для этого набора корректный результат — **30**
- `(7 + 5 - 3) * 5 * 5 - 5 = 220`, поэтому для этого набора корректный результат — **220**

Таким образом, окончательная таблица:


| a | b | c | FASM |
|---:|---:|---:|---:|
| 2 | 3 | 4 | 44 |
| 5 | 2 | 3 | 51 |
| 10 | 4 | 2 | 30 |
| 7 | 3 | 5 | 220 |


## 6. Вывод

В ходе лабораторной работы была разработана программа на **FASM** для получения ASCII-кода символа, переданного через командную строку.

Для **варианта 19** реализовано арифметическое выражение:

$$(((((a+c)-b)\cdot c)\cdot c)-c)$$

Все операции, включая разбор параметров командной строки, вычисление выражения и вывод результата, реализованы непосредственно на FASM с использованием системных вызовов Linux.

Таким образом, для выполнения лабораторной работы не используются NASM, C, `printf`, `atoi` или другие внешние библиотеки.
